# Nemotron Voice Agent

Nemotron Voice Agent Blueprint provides a comprehensive, end-to-end voice agent built with NVIDIA Nemotron state-of-the-art open models. It guides developers through building a cascaded pipeline that integrates ASR, LLM, and TTS while handling streaming, interruptible conversations. Clone it, add your logic, and deploy a working voice AI prototype.

This Brev launchable deploys either a local **server** recipe, which uses NVIDIA NIM services, or a local **single-GPU** recipe, which uses vLLM and NeMo-Speech.cpp. Select exactly one recipe family in the **Setup** section.

**GitHub Repository:** [NVIDIA-AI-Blueprints/nemotron-voice-agent](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent)

## Why This Blueprint

- **Sub-second E2E latency:** sub-second end-to-end latency with support for multiple concurrent streams, designed for production scale.
- **Fully open models:** Nemotron Streaming and Parakeet ASR, Magpie TTS and Nemotron LLM models. Swap any component, self-host, no lock-in.
- **Interruption Handling:** Voice Activity Detection (VAD) and End-of-Utterance (EOU) logic to guide the agent on exactly when to start and stop speaking, ensuring a natural conversational flow.
- **Multilingual Capabilities:** native support for multiple languages provided by NVIDIA Magpie TTS and Multilingual ASR.
- **Multimodal Understanding:** reason over speech and vision together, analyzing live camera input and uploaded media (images, documents) within a single conversation, powered by Nemotron Omni.
- **Multi-Agent and Tool Calling:** orchestrate cooperating agents that invoke external tools and functions for task-oriented workflows, while decoupling reasoning from response generation for lower perceived latency.
- **Edge Support:** deploy anywhere, from cloud and workstation to DGX Spark and edge devices like Jetson Thor, using self-contained deployment recipes.

---

## Architecture

![Nemotron Voice Agent Architecture](https://raw.githubusercontent.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/main/docs/images/arch.png)

---

## Available Examples

Review the examples below and set `EXAMPLE_PROFILE` in the **Setup** section to one listed profile. The default is `generic-assistant/single-gpu`. `*/server` profiles use local NIM services and are workstation-only. `*/single-gpu` profiles use vLLM and NeMo-Speech.cpp; they support compatible workstations and DGX Spark, while Generic, Multilingual, Omni, and Frontend/Backend also support Jetson Thor.

| Example | Profile values | Description |
|---------|----------------|-------------|
| [Generic Assistant](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/generic/README.md) | `generic-assistant/server` or `generic-assistant/single-gpu` | English cascaded pipeline. The single-GPU profile is the best starting point. |
| [Multilingual Assistant](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/multilingual/README.md) | `multilingual-assistant/server` or `multilingual-assistant/single-gpu` | Multilingual ASR and TTS with a fixed language per session. |
| [Omni Assistant](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/omni_assistant/README.md) | `omni-assistant/server` or `omni-assistant/single-gpu` | A single Nemotron Omni model replaces ASR and LLM stages. |
| [Omni Subagents](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/omni_assistant_subagents/README.md) | `omni-assistant-subagents/server` or `omni-assistant-subagents/single-gpu` | Multi-agent Omni pipeline for richer audio, video, and webcam understanding. The single-GPU profile is not supported on Jetson Thor. |
| [Frontend/Backend Agent](https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/blob/main/src/examples/frontend_backend_agent/README.md) | `frontend-backend-agent/server` or `frontend-backend-agent/single-gpu` | Fast frontend LLM and specialized backend agent. Add voice to an existing text or agentic backend. |

---


## 1. Prerequisites

Before running this notebook, confirm your environment has:

| Requirement | Details |
|-------------|--------|
| **GPU** | **Server:** a compatible workstation with capacity for the selected NIM stack; server profiles do not support DGX Spark or Jetson Thor. **Single-GPU:** one compatible GPU; confirm memory fits the selected vLLM and speech recipe. |
| **Docker** | Docker Engine with NVIDIA GPU support and Docker Compose v2.20+ |
| **Hugging Face token** | Required as `HF_TOKEN` for `*/single-gpu` profiles to download vLLM and NeMo-Speech.cpp model weights. Create one at [Hugging Face settings](https://huggingface.co/settings/tokens). |
| **NVIDIA API Key** | Required as `NVIDIA_API_KEY` for `*/server` profiles. It authorizes NIM services and the non-interactive NGC container-registry login. The notebook keeps reading this key for server use; single-GPU profiles do not use it. |
| **Network ports** | Port `7860` for the web app; `3478` and `49160–49200` UDP/TCP for WebRTC TURN |

Select exactly one profile. Do not select `omni-assistant-subagents/single-gpu` on Jetson Thor.

---


## 2. Setup

### 2a. Clone the Repository

This cell clones the Nemotron Voice Agent repository if it is not already present.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/NVIDIA-AI-Blueprints/nemotron-voice-agent.git"
CLONE_DIR = Path(os.path.expanduser("~/nemotron-voice-agent"))

if not CLONE_DIR.exists():
    print(f"Cloning {REPO_URL} → {CLONE_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    print("Clone complete.")
else:
    print(f"Repository already present at {CLONE_DIR}. Pulling latest changes.")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

REPO_ROOT = CLONE_DIR
print(f"Repo root: {REPO_ROOT}")

### 2b. Configure Credentials and Deployment Settings

The notebook prompts only for the credential required by the selected profile. `*/server` requires `NVIDIA_API_KEY` and uses it for NIM plus the NGC login. `*/single-gpu` requires `HF_TOKEN` for model downloads; it does not use `NVIDIA_API_KEY` or log in to `nvcr.io`.

---


In [ ]:
import getpass

# ── Example to deploy ──────────────────────────────────────────────────────────
# Server profiles use local NIM services and are workstation-only.
SERVER_PROFILES = (
    "generic-assistant/server",
    "multilingual-assistant/server",
    "omni-assistant/server",
    "omni-assistant-subagents/server",
    "frontend-backend-agent/server",
)

# Single-GPU profiles use vLLM and NeMo-Speech.cpp.
# omni-assistant-subagents/single-gpu is not supported on Jetson Thor.
SINGLE_GPU_PROFILES = (
    "generic-assistant/single-gpu",
    "multilingual-assistant/single-gpu",
    "omni-assistant/single-gpu",
    "omni-assistant-subagents/single-gpu",
    "frontend-backend-agent/single-gpu",
)
SUPPORTED_PROFILES = SERVER_PROFILES + SINGLE_GPU_PROFILES

EXAMPLE_PROFILE = os.environ.get("EXAMPLE_PROFILE", "generic-assistant/single-gpu")
if EXAMPLE_PROFILE not in SUPPORTED_PROFILES:
    raise ValueError(
        f"EXAMPLE_PROFILE must be one of: {', '.join(SUPPORTED_PROFILES)}"
    )

IS_SERVER_PROFILE = EXAMPLE_PROFILE in SERVER_PROFILES
IS_SINGLE_GPU_PROFILE = EXAMPLE_PROFILE in SINGLE_GPU_PROFILES

# ── Credentials (prompted only when the selected profile needs them) ───────────
# Keep reading NVIDIA_API_KEY: it is required for */server and harmless when set
# for */single-gpu, where it is not sent to NVIDIA services or NGC.
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Advanced settings ──────────────────────────────────────────────────────────
PIPELINE_APP_PORT = os.environ.get("PIPELINE_APP_PORT", "7860")
PIPELINE_TLS = "false"  # Brev Tunnels terminate HTTPS → set to "true" for direct HTTPS
RESET_VOLUMES = False  # Set True to wipe Docker, Hugging Face, and model caches
AUTO_DETECT_TURN = True  # Auto-detect public IP for TURN URL

recipe_family = "server (NIM)" if IS_SERVER_PROFILE else "single-gpu (vLLM + NeMo-Speech.cpp)"
print(f"Example profile : {EXAMPLE_PROFILE}")
print(f"Recipe family   : {recipe_family}")
print(f"App port        : {PIPELINE_APP_PORT}")
print(f"TLS             : {PIPELINE_TLS}")
print()

if IS_SERVER_PROFILE:
    if not NVIDIA_API_KEY:
        NVIDIA_API_KEY = getpass.getpass(
            "🔑 NVIDIA_API_KEY (required for the server profile): "
        ).strip()
    if not NVIDIA_API_KEY:
        raise RuntimeError(
            "NVIDIA_API_KEY is required for */server NIM services and the NGC login. "
            "Get a key at https://build.nvidia.com/"
        )
else:
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass(
            "🤗 HF_TOKEN (required for the single-GPU recipe): "
        ).strip()
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN is required to download the vLLM and NeMo-Speech.cpp model weights. "
            "Get a token at https://huggingface.co/settings/tokens"
        )
    if NVIDIA_API_KEY:
        print("NVIDIA_API_KEY is retained in .env but is not used by this single-GPU deployment.")
print("Credentials configured.")


### 2c. Initialize Utilities and `.env`

In [ ]:
import json
import secrets
import subprocess
import time
import urllib.request

ENV_PATH = REPO_ROOT / ".env"
ENV_EXAMPLE_PATH = REPO_ROOT / ".env.example"
PROFILES = [EXAMPLE_PROFILE, "turn"]
DOCKER = ["docker"]
COMPOSE = [*DOCKER, "compose"]
COMPOSE_PROGRESS = [*DOCKER, "compose", "--progress", "plain"]
for _p in PROFILES:
    COMPOSE.extend(["--profile", _p])
    COMPOSE_PROGRESS.extend(["--profile", _p])


def redact(text: str) -> str:
    """Replace secret values with a redaction marker before printing."""
    for v in (NVIDIA_API_KEY, HF_TOKEN):
        if v:
            text = text.replace(v, "***REDACTED***")
    return text


def run(cmd, *, check=True, input_text=None, stream=False):
    """Run a shell command from the repo root and print redacted output."""
    print("$ " + redact(" ".join(cmd)))
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            stdin=subprocess.PIPE if input_text else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if input_text and proc.stdin:
            proc.stdin.write(input_text)
            proc.stdin.close()
        out = []
        for line in proc.stdout:
            out.append(line)
            stripped = line.rstrip()
            if not stripped.endswith("Pulling fs layer 0B"):
                print(redact(stripped))
        proc.wait()
        stdout = "".join(out)
        if check and proc.returncode != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd, stdout, "")
        return subprocess.CompletedProcess(cmd, proc.returncode, stdout, "")
    proc = subprocess.run(cmd, cwd=REPO_ROOT, input=input_text, text=True, capture_output=True)
    if proc.stdout:
        print(redact(proc.stdout.rstrip()))
    if proc.stderr:
        print(redact(proc.stderr.rstrip()))
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, proc.stdout, proc.stderr)
    return proc


def fetch_text(url, timeout=10):
    """Fetch a URL and return the response body as text."""
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode("utf-8", errors="replace")


def detect_public_ip():
    """Return the public IP of this instance."""
    return urllib.request.urlopen("https://ifconfig.me", timeout=10).read().decode().strip()


def read_env(path):
    """Read an env file into a list of lines."""
    return path.read_text().splitlines() if path.exists() else []


def get_env(lines, key):
    """Return the value for a key from parsed env lines."""
    for line in lines:
        s = line.strip()
        if s.startswith(f"{key}="):
            return s.split("=", 1)[1]
    return ""


def set_env(lines, key, value):
    """Set or append a key=value in parsed env lines."""
    replacement = f"{key}={value}"
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith(f"{key}=") or s.startswith(f"# {key}=") or s.startswith(f"#{key}="):
            lines[i] = replacement
            return lines
    lines.append(replacement)
    return lines


# Bootstrap .env from example if missing
if not ENV_PATH.exists():
    ENV_PATH.write_text(ENV_EXAMPLE_PATH.read_text())

lines = read_env(ENV_PATH)

TURN_USERNAME = os.environ.get("TURN_USERNAME") or get_env(lines, "TURN_USERNAME") or "admin"
TURN_PASSWORD = os.environ.get("TURN_PASSWORD") or get_env(lines, "TURN_PASSWORD") or ""
TURN_URL = os.environ.get("TURN_URL") or get_env(lines, "TURN_URL") or ""

if not TURN_PASSWORD or TURN_PASSWORD == "admin":
    TURN_PASSWORD = secrets.token_urlsafe(32)
    print("Generated random TURN_PASSWORD.")

if AUTO_DETECT_TURN and (not TURN_URL or "brevlab.com" in TURN_URL):
    _ip = detect_public_ip()
    TURN_URL = f"turn:{_ip}:3478"
    print(f"Auto-detected TURN_URL: {TURN_URL}")

env_values = {
    "NVIDIA_API_KEY": NVIDIA_API_KEY,
    "HF_TOKEN": HF_TOKEN,
    "TURN_USERNAME": TURN_USERNAME,
    "TURN_PASSWORD": TURN_PASSWORD,
    "TURN_URL": TURN_URL,
    "PIPELINE_APP_PORT": PIPELINE_APP_PORT,
    "PIPELINE_TLS": PIPELINE_TLS,
}
for k, v in env_values.items():
    if v:
        lines = set_env(lines, k, v)
ENV_PATH.write_text("\n".join(lines).rstrip() + "\n")
os.environ.update({k: v for k, v in env_values.items() if v})
print(f"\n.env written: {ENV_PATH}")
print(f"Profiles     : {', '.join(PROFILES)}")

### 2d. Download NeMo-Speech.cpp Models (Single-GPU Only)

For a `*/single-gpu` profile, this cell downloads the ASR and TTS model files used by NeMo-Speech.cpp. It reads `HF_TOKEN` from `.env`; run it as the Brev user, not as root. For a `*/server` profile the cell skips this step because the profile uses NIM containers instead.

---


In [ ]:
if IS_SINGLE_GPU_PROFILE:
    run(["bash", "scripts/download-nemo-speech-models.sh"], stream=True)
else:
    print("Skipping NeMo-Speech.cpp model download: the server profile uses NIM containers.")


---

## 3. Preflight Checks

Verify Docker, GPU visibility, disk space, and the selected Compose profile before pulling containers.

In [ ]:
run([*DOCKER, "--version"])
run([*DOCKER, "compose", "version"])
run([*DOCKER, "ps"])
run([*DOCKER, "info", "--format", "DockerRootDir={{.DockerRootDir}} runtime={{.DefaultRuntime}}"], check=False)
run(["df", "-h", str(REPO_ROOT)], check=False)
run(["nvidia-smi"])
run([*COMPOSE, "config", "--profiles"])

---

## 4. Deploy

For a `*/server` profile, the notebook signs in to `nvcr.io` with `NVIDIA_API_KEY`, then pulls the NIM services and application. For a `*/single-gpu` profile, it starts the vLLM and NeMo-Speech.cpp sidecars without an NGC login; NeMo-Speech.cpp weights were downloaded earlier and vLLM downloads its model weights on first start. First startup can take up to 30 minutes; later starts reuse caches.

Safe to re-run: containers are recreated, and model-cache volumes are preserved unless `RESET_VOLUMES=True`.

---


In [ ]:
# Authenticate before changing an existing deployment, so a server-profile
# credential error leaves the currently running containers untouched.
if IS_SERVER_PROFILE:
    run(
        [*DOCKER, "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
        input_text=f"{NVIDIA_API_KEY}\n",
    )
else:
    print("Skipping NGC login: the single-GPU profile does not use nvcr.io.")

# Tear down any existing deployment.
down_cmd = [*DOCKER, "compose", "--profile", "*", "down", "--remove-orphans"]
if RESET_VOLUMES:
    down_cmd.append("-v")
    print("RESET_VOLUMES=True: model cache volumes will be deleted.")
run(down_cmd, check=False, stream=True)

# Pull and start (--quiet-pull suppresses per-layer download noise).
run([*COMPOSE_PROGRESS, "up", "-d", "--build", "--remove-orphans", "--quiet-pull"], stream=True)
run([*COMPOSE, "ps"])


---

## 5. Accessing the Application

### Health check

The cell below polls until the application is ready, up to 5 minutes. This indicates that the web app is reachable. NIM services or vLLM and NeMo-Speech.cpp can continue loading after the app responds, especially on the first start. Confirm their status with `docker compose ps` before starting a voice session.


In [ ]:
scheme = "http" if PIPELINE_TLS.lower() == "false" else "https"
health_url = f"{scheme}://localhost:{PIPELINE_APP_PORT}/health"
ice_url = f"{scheme}://localhost:{PIPELINE_APP_PORT}/api/ice-servers"

for attempt in range(1, 31):
    try:
        fetch_text(health_url, timeout=5)
        print(f"✅ App is healthy: {health_url}")
        break
    except Exception as exc:
        if attempt == 30:
            raise
        print(f"Waiting for app ({attempt}/30): {exc}")
        time.sleep(10)

try:
    ice = json.loads(fetch_text(ice_url, timeout=10))
    print("ICE / TURN config:")
    print(json.dumps(ice, indent=2))
except Exception as exc:
    print(f"ICE check failed (non-fatal): {exc}")

### Open the Web UI

**Brev Tunnel** (remote access through the Brev proxy):
1. In the Brev console, open the instance.
2. Under **Access**, find **Using Tunnels**.
3. Add port `7860`, then open or copy the generated URL.

Once connected, you will see the Nemotron Voice Agent interface:

![Nemotron Voice Agent UI](https://raw.githubusercontent.com/NVIDIA-AI-Blueprints/nemotron-voice-agent/main/docs/images/ui.png)

Press **Connect** to start a voice conversation.

### Tips

- **Audio:** use a wired headset for best results.
- **Warm-up:** the first voice turn can be slower while the selected NIM services or local vLLM and NeMo-Speech.cpp sidecars finish loading. Subsequent turns are much faster.
- **Logs:** run `docker compose logs -f` in the repo root to tail all service logs.

---

### Troubleshooting

**WebRTC not connecting?**

Configure these exposed TCP/UDP ports in the Brev Launchable **Network** section:
- Port `3478` (TCP + UDP): TURN control
- Range `49160-49200` (UDP): TURN relay

> The notebook auto-detects the instance public IP for `TURN_URL`. Do **not** use the Brev Tunnel hostname as `TURN_URL`. TURN requires a direct UDP/TCP path to the instance IP.
